# View Purge Behavior — `purge-view-metadata-on-drop`

> **New here? Read this first.** This notebook answers a precise question a manager
> raised: *what happens when you drop a VIEW under each value of the catalog config
> `polaris.config.purge-view-metadata-on-drop`?*

## Background

A view holds no data files — only a small `.gz.metadata.json`. The question is what a
view DROP does to that metadata file under each config, and whether the
`drop-with-purge.enabled` flag changes the outcome.

```
purge-view-metadata-on-drop = true   → drop SHOULD delete the view's metadata file
purge-view-metadata-on-drop = false  → drop is metadata-only (file remains by design)
drop-with-purge.enabled              → gates table purge; does it gate VIEW drop too?
```

In our no-STS environment, even when a view drop is supposed to delete the metadata
file, the file is **orphaned** — the same async-cleanup-task failure that affects
table purge (see `polaris-purge-test-report.md`, Apache Polaris #379).

## What this notebook tests (each case creates its OWN fresh view)

```
Case 1   purge-view-metadata=true  + drop-with-purge=false
Case 2   purge-view-metadata=true  + drop-with-purge=true
Case 3   purge-view-metadata=false
→ record the drop status code AND whether the metadata file remains, for each.
The truth table is assembled from OBSERVED results (not assumed).
```

> **Note:** an earlier exploratory session suggested `purge-view-metadata=true` +
> `drop-with-purge=false` returns 403. This notebook tests that claim directly rather
> than assuming it — let the observed status codes decide.

## Prerequisites & how to run

```
Files:   polaris_test_utils.py, minio_rest.py, config/  (same folder)
Packages: pip install requests pyyaml opensearch-py
Run:     run the setup cell (prints 🟢 DEV / 🔴 PROD), then top to bottom.
Switch:  change init_env("dev") → init_env("prod") in setup, re-run that cell.

⚠️  Storage access via MinIO REST API (minio_rest.py) — NO s3fs. No hardcoded creds.
    The real view metadata is a >0-byte .gz.metadata.json inside metadata/.
    mc.list filters 0-byte directory markers, so view_files() counts only real files.
```

---


In [2]:
# ── Setup ─────────────────────────────────────────────────────────────────
# Environment switch HERE. Banner shows 🟢 DEV / 🔴 PROD. Change "dev" → "prod"
# and re-run THIS cell to switch (no kernel restart needed).
from polaris_test_utils import *
init_env("dev")

import uuid, time
tok = root_token()

TEST_CATALOG_V = "view-purge-test"
NS = "vp-ns"

def view_files(view_name):
    """Recursive list of REAL (>0 byte) files under a view's location."""
    return mc.list(f"{TEST_CATALOG_V}/{NS}/{view_name}/")   # mc.list filters 0-byte markers

def create_view(catalog, ns, name):
    return requests.post(f"{BASE_CAT}/{catalog}/namespaces/{ns}/views", headers=h(tok),
        json={"name": name, "default-namespace": [ns],
              "schema": {"type":"struct","schema-id":0,
                         "fields":[{"id":1,"name":"id","type":"long","required":True}]},
              "view-version": {"version-id":1,"timestamp-ms":int(time.time()*1000),"schema-id":0,
                               "default-namespace":[ns],"summary":{"engine":"spark"},
                               "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
              "location": f"s3a://{BUCKET}/{TEST_CATALOG_V}/{ns}/{name}/"})

def set_catalog_config(catalog, **flags):
    cat = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}), **{f"polaris.config.{k.replace('_','-')}":v for k,v in flags.items()}}
    requests.put(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})

print("✅ setup ready")


🟢 DEV   POLARIS_ENV=dev   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ setup ready


In [3]:
# Cleanup any leftover, then create a fresh catalog + namespace.
# This is HARDENED: it drops views AND tables AND nested namespaces, checks each
# delete, and VERIFIES the catalog is gone before recreating (a stale catalog with
# old config flags would otherwise corrupt the test — that's what caused earlier
# wrong results).

def force_delete_catalog(cat):
    # must exist?
    if requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code == 404:
        return True
    # enable purge + grant so we can delete everything
    set_catalog_config(cat, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
    grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)

    # list namespaces DEEPEST-first (so child namespaces go before parents)
    nss = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
    nss = sorted(nss, key=lambda n: len(n if isinstance(n, list) else [n]), reverse=True)
    for ns in nss:
        nsn_simple = ns[0] if isinstance(ns, list) else ns
        # drop tables
        for t in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn_simple}/tables", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn_simple}/tables/{t['name']}?purgeRequested=true", headers=h(tok))
        # drop views
        for v in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn_simple}/views", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn_simple}/views/{v['name']}?purgeRequested=true", headers=h(tok))
        # drop namespace
        rns = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn_simple}", headers=h(tok))
        if rns.status_code not in (200, 204):
            print(f"   ⚠️ namespace {nsn_simple} delete: {rns.status_code} {rns.text[:80]}")
    # delete catalog
    rc = requests.delete(f"{BASE_MGMT}/catalogs/{cat}?purgeRequested=true", headers=h(tok))
    # sweep storage
    mc.delete_prefix(f"{cat}/", min_size=0)
    # VERIFY gone
    still = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code != 404
    if still:
        print(f"   ❌ catalog {cat} STILL EXISTS after delete (status {rc.status_code}): {rc.text[:120]}")
        return False
    return True

print("Cleanup leftover...")
ok = force_delete_catalog(TEST_CATALOG_V)
print(f"  catalog removed: {ok}")

# Create fresh catalog ONLY if the old one is gone
if not ok:
    raise RuntimeError(f"Could not remove stale catalog {TEST_CATALOG_V}. "
                       f"Resolve manually before re-running (stale config would corrupt the test).")

r = create_catalog(TEST_CATALOG_V, token=tok,
                   properties={"polaris.config.purge-view-metadata-on-drop":"true"})
if r.status_code not in (200, 201):
    print(f"  ❌ create_catalog failed: {r.status_code} {r.text[:150]}")
    raise RuntimeError("catalog creation failed")
grant_privilege(catalog=TEST_CATALOG_V, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)
create_namespace(catalog=TEST_CATALOG_V, ns=NS, token=tok)
print("✅ FRESH catalog + namespace ready (verified clean)")


Cleanup leftover...
   ⚠️ namespace vp-ns delete: 409 {"error":{"message":"Namespace vp-ns is not empty","type":"NamespaceNotEmptyExce
   ❌ catalog view-purge-test STILL EXISTS after delete (status 400): {"error":{"message":"Catalog 'view-purge-test' cannot be dropped, it is not empty","type":"BadRequestException","code":4
  catalog removed: False


RuntimeError: Could not remove stale catalog view-purge-test. Resolve manually before re-running (stale config would corrupt the test).

## Test runner — each case creates its own view, drops it, checks the file

`run_case` makes a fresh view, sets the two config flags, drops the view, and reports
the drop status code plus whether the real metadata file survived. Self-contained so
the three cases don't interfere with each other.


In [ ]:
results = {}

def run_case(case_name, purge_view_metadata, drop_with_purge):
    vname = f"view-{case_name}-{uuid.uuid4().hex[:6]}"
    # set flags BEFORE creating/dropping
    set_catalog_config(TEST_CATALOG_V,
                       purge_view_metadata_on_drop=str(purge_view_metadata).lower(),
                       drop_with_purge_enabled=str(drop_with_purge).lower())
    create_view(TEST_CATALOG_V, NS, vname)
    before = len(view_files(vname))

    r = requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{NS}/views/{vname}", headers=h(tok))
    gone = requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{NS}/views/{vname}",
                        headers=h(tok)).status_code == 404
    time.sleep(2)
    after = len(view_files(vname))

    results[case_name] = {"status": r.status_code, "view_gone": gone,
                          "file_before": before, "file_after": after,
                          "err": (r.text[:120] if r.status_code >= 400 else "")}
    print(f"[{case_name}] purge-view-metadata={purge_view_metadata}, drop-with-purge={drop_with_purge}")
    print(f"   drop status: {r.status_code}   view gone: {gone}")
    print(f"   metadata file: {before} -> {after}", end="")
    if r.status_code >= 400:
        print(f"   (err: {r.text[:80]})")
    elif after >= before and before > 0:
        print("   → file ORPHANED (drop succeeded, file not deleted)")
    elif after == 0 and before > 0:
        print("   → file DELETED")
    else:
        print()
    print()
    return results[case_name]


In [ ]:
# Case 1 — purge-view-metadata=true + drop-with-purge=false
run_case("case1", purge_view_metadata=True, drop_with_purge=False)


In [ ]:
# Case 2 — purge-view-metadata=true + drop-with-purge=true
run_case("case2", purge_view_metadata=True, drop_with_purge=True)


In [ ]:
# Case 3 — purge-view-metadata=false
run_case("case3", purge_view_metadata=False, drop_with_purge=True)


## Truth table — observed results

In [ ]:
print("="*70)
print(f"VIEW DROP BEHAVIOR — observed results (Polaris {POLARIS_VERSION})")
print("="*70)
print()
print(f"{'case':7} {'purge-view':11} {'drop-purge':11} {'status':7} {'file':12} {'gone'}")
print("-"*60)
spec = {"case1": ("true","false"), "case2": ("true","true"), "case3": ("false","true")}
for cse, (pvm, dwp) in spec.items():
    r = results.get(cse, {})
    if not r:
        print(f"{cse:7} {pvm:11} {dwp:11} (not run)")
        continue
    fb, fa = r["file_before"], r["file_after"]
    file_outcome = ("orphaned" if fa>=fb and fb>0 else
                    "deleted" if fa==0 and fb>0 else f"{fb}->{fa}")
    print(f"{cse:7} {pvm:11} {dwp:11} {r['status']:<7} {file_outcome:12} {r['view_gone']}")
print()
print("Interpretation (from the observed rows above):")
# derive the coupling conclusion from data
c1 = results.get("case1", {})
if c1.get("status") in (200,204):
    print("  • drop-with-purge=false did NOT block the view drop (got "
          f"{c1.get('status')}, not 403).")
    print("    → the earlier '403 coupling' hypothesis is NOT confirmed in this env.")
elif c1.get("status") == 403:
    print("  • drop-with-purge=false BLOCKED the view drop (403).")
    print("    → confirms the coupling: purge-view-metadata=true needs drop-with-purge=true.")
print("  • Where a metadata file shows 'orphaned', the drop succeeded but the file")
print("    was NOT deleted (no-STS cleanup-task failure, Polaris #379).")


## Cleanup

In [ ]:
set_catalog_config(TEST_CATALOG_V, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces", headers=h(tok)).json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
    requests.delete(f"{BASE_CAT}/{TEST_CATALOG_V}/namespaces/{nsn}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG_V}?purgeRequested=true", headers=h(tok))
mc.delete_prefix(f"{TEST_CATALOG_V}/")
print("✅ cleaned up")
